# Decay basics with pydecay

Work through the exponential decay law with the bundled ICRP-107 nuclide
catalog: load half-lives, decay an activity over time, convert units, and
pin every number against a known-good golden value.

**Goals**

1. Understand `lambda = ln(2) / T_half` and `A(t) = A0 * exp(-lambda * t)`.
2. Load nuclides and read their half-lives.
3. Decay activities and fractions with `decayed_activity` / `remaining_fraction`.
4. Convert between becquerel and curie.

**Sources**

* ICRP Publication 107 (nuclide half-lives): https://www.icrp.org/publication.asp?id=ICRP+Publication+107
* NIST Special Publication 811 (unit definitions, `1 Ci = 3.7e10 Bq` exact): https://www.nist.gov/pml/special-publication-811


## The exponential decay law

A single radionuclide with half-life $T_{1/2}$ decays with constant decay
constant

$$\lambda = \frac{\ln 2}{T_{1/2}}, \qquad N(t) = N_0 e^{-\lambda t}, \qquad
A(t) = \lambda N(t) = A_0 e^{-\lambda t}.$$

One half-life halves the population, two half-lives quarter it, and so on.
The mean lifetime is $\tau = 1/\lambda$. Every half-life used below comes
from the bundled ICRP-107 catalog (ICRP Publication 107,
https://www.icrp.org/publication.asp?id=ICRP+Publication+107).


In [ ]:
import pydecay
from pydecay import Nuclide, bq_to_ci, decayed_activity, remaining_fraction

print("pydecay", pydecay.__version__)

i131 = Nuclide.load("I-131")
co60 = Nuclide.load("Co-60")
c14 = Nuclide.load("C-14")

print(f"I-131  T_half = {i131.half_life_s:>15.2f} s = {i131.half_life_s / 86400:.4f} days")
print(f"Co-60  T_half = {co60.half_life_s:>15.2f} s = {co60.half_life_s / 31557600:.4f} years")
print(f"C-14   T_half = {c14.half_life_s:>15.2f} s = {c14.half_life_s / 31557600:.0f} years")


## Units: seconds in, seconds out

pydecay accepts plain floats (seconds) or unit strings that pint understands
(`"6 hours"`, `"30 days"`). The golden tests and this notebook use the
**Julian year** `31557600 s = 365.25 days` wherever "years" appear, so the
numbers are reproducible to the last digit.

For activity, `1 Ci = 3.7e10 Bq` exactly (NIST Special Publication 811,
https://www.nist.gov/pml/special-publication-811). Bq, Ci, and hour/day
strings all work through the same conversion chokepoint.


In [ ]:
a0_mbq = 1e6  # 1 MBq of Co-60
t_10y = 10 * 31557600  # 10 Julian years in seconds

a_10y = decayed_activity(a0_mbq, co60.half_life, t_10y)
print(f"Co-60 after 10 years: {a_10y:.6f} Bq  (from 1e6 Bq)")

tc99m = Nuclide.load("Tc-99m")
frac_6h = remaining_fraction(tc99m.half_life, "6 hours")
print(f"1e6 Bq = {bq_to_ci(1e6):.6e} Ci")
print(f"6-hour Tc-99m remaining fraction: {frac_6h:.6f}")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

days = np.arange(0, 31, 0.5)
activity = [decayed_activity(1.0, i131.half_life, d * 86400) for d in days]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(days, activity, label="I-131")
ax.axhline(0.5, color="grey", ls="--", lw=1, label="half activity")
ax.set_xlabel("time (days)")
ax.set_ylabel("A(t) / A0")
ax.set_title("I-131 decay curve (ICRP-107 half-life)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## Exercise: decay a Tc-99m eluate

A generator eluate starts at **2.5e5 Bq** of Tc-99m (half-life 6.015 hours).
How much activity remains after **6 hours**?

Compute it two ways (`decayed_activity`, or `remaining_fraction` multiplied
by the start) and check the answers agree. The solution is in the next cell,
tagged `answer` - hide it first if you want to try unaided.


In [ ]:
# ANSWER (cell tagged "answer" - hide while attempting the exercise)
hl_tc = Nuclide.load("Tc-99m").half_life
way1 = decayed_activity(2.5e5, hl_tc, "6 hours")
way2 = 2.5e5 * remaining_fraction(hl_tc, "6 hours")
print(f"after 6 h: {way1:.6f} Bq  vs  {way2:.6f} Bq")
assert abs(way1 - way2) < 1e-6
# Same fraction the golden test pins for 500 Bq -> 250.43251034568436 Bq:
assert remaining_fraction(hl_tc, 21600) == 250.43251034568436 / 500
print("exercise checks passed")


## Golden check

The values below are pinned in `tests/golden/golden_values.json`:
ICRP-107 half-lives for I-131 and Co-60, the analytic 10-year Co-60
activity `A0*exp(-ln2/T*t)`, and the C-14 remaining fraction after
5730 Julian years. `tests/test_notebooks.py` re-checks every printed value
against that file, so this notebook fails if the data or the math drifts.


In [ ]:
import json
import math

hl_i131 = i131.half_life_s
hl_co60 = co60.half_life_s
a10 = decayed_activity(1e6, co60.half_life, 10 * 31557600)
c14_frac = remaining_fraction(c14.half_life, 5730 * 31557600)

assert math.isclose(hl_i131, 692988.48, rel_tol=1e-12)
assert math.isclose(hl_co60, 166349576.88, rel_tol=1e-12)
assert math.isclose(a10, 268488.97629773617, rel_tol=1e-9)
assert math.isclose(c14_frac, 0.49817925166675786, rel_tol=1e-9)

payload = {
    "i131_halflife_iaea": hl_i131,
    "co60_halflife_iaea": hl_co60,
    "co60_activity_10y": a10,
    "c14_fraction_5730y": c14_frac,
}
print("GOLDEN_CHECK " + json.dumps(payload))
